# Etapa 6 — Lógica Fuzzy, Integração Final e Validação com 2025

O sistema Mamdani (scikit-fuzzy) recebe:
- **presença** = P(facção | porte, fronteira, sintomas), vinda da rede bayesiana (Etapa 5);
- **tendência** = variação relativa dos homicídios de 2020–21 para 2022–23,

e devolve a **prioridade de monitoramento / inteligência policial** (0–100).

**Regras**
1. SE presença é alta E (tendência é alta OU estável) ENTÃO prioridade é alta.
2. SE presença é alta E tendência é queda ENTÃO prioridade é média (possível **domínio consolidado**).
3. SE presença é média E tendência é alta ENTÃO prioridade é alta (possível **expansão**).
4. SE presença é média E (tendência é estável OU queda) ENTÃO prioridade é média.
5. SE presença é baixa ENTÃO prioridade é baixa.

Depois conferimos a decisão com dois gabaritos: a presença em **2024** (o rótulo usado no treino) e as **entradas em 2025** (4ª edição do Cartografias), que o modelo nunca viu.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 115)


In [2]:
from fuzzy import criar_sistema, avaliar, rotulo
probs = pd.read_csv("dados/probabilidades_bayesianas.csv")
sistema, (presenca, tendencia, prioridade) = criar_sistema()
fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for k, var in enumerate([presenca, tendencia, prioridade]):
    for nome, termo in var.terms.items():
        ax[k].plot(var.universe, termo.mf, label=nome)
    ax[k].set_title(var.label); ax[k].legend(fontsize=7)
plt.tight_layout(); plt.savefig("figuras/etapa6_pertinencia.png", dpi=150)

In [3]:
probs["tend_cap"] = probs["variacao_hom"].clip(-1, 2)
probs["prioridade"] = [avaliar(sistema, p, t) for p, t in zip(probs["p_faccao"], probs["tend_cap"])]
probs["decisao_fuzzy"] = probs["prioridade"].map(rotulo)
probs["decisao_rigida"] = np.where(probs["p_faccao"] >= 0.5, "alta", "baixa")
print(probs["decisao_fuzzy"].value_counts()); print(probs["decisao_rigida"].value_counts())

decisao_fuzzy
baixa    404
alta     239
média    165
Name: count, dtype: int64
decisao_rigida
baixa    574
alta     234
Name: count, dtype: int64


## 1. Decisão fuzzy × decisão rígida, conferidas com o Cartografias 2024

In [4]:
def resumo(col):
    g = probs.groupby(col)
    return pd.DataFrame({"municípios": g.size(), "com facção 2024": g["ALVO"].mean(),
                         "em disputa 2024": g["REGIME"].apply(lambda s: (s == 2).mean())})
print("Decisão fuzzy:"); print(resumo("decisao_fuzzy").reindex(["baixa", "média", "alta"]).round(3))
print("\nDecisão rígida (P >= 0,5):"); print(resumo("decisao_rigida").round(3))

Decisão fuzzy:
               municípios  com facção 2024  em disputa 2024
decisao_fuzzy                                              
baixa                 404            0.129            0.017
média                 165            0.376            0.103
alta                  239            0.607            0.255

Decisão rígida (P >= 0,5):
                municípios  com facção 2024  em disputa 2024
decisao_rigida                                              
alta                   234            0.641            0.274
baixa                  574            0.190            0.037


## 2. Validação fora do tempo: o alerta antecipa a expansão de 2025?

Olhamos só os municípios **sem facção em 2024** e conferimos quantos passaram a ter facção em **2025**, segundo a 4ª edição do Cartografias.

In [5]:
from sklearn.metrics import roc_auc_score
sem = probs[probs["ALVO"] == 0].copy()
print(f"Municípios sem facção em 2024: {len(sem)}  |  passaram a ter em 2025: {int(sem['ALVO_2025'].sum())} ({sem['ALVO_2025'].mean():.1%})")
print("\nEntrada em 2025 por prioridade fuzzy (calculada só com dados até 2023):")
print(sem.groupby("decisao_fuzzy")["ALVO_2025"].agg(municipios="size", entraram="mean").reindex(["baixa", "média", "alta"]).round(3))
print("\nEntrada em 2025 pela decisão rígida:")
print(sem.groupby("decisao_rigida")["ALVO_2025"].agg(municipios="size", entraram="mean").round(3))
print(f"\nAUC da probabilidade bayesiana para prever a entrada em 2025: {roc_auc_score(sem['ALVO_2025'], sem['p_faccao']):.3f}")
print(f"AUC da prioridade fuzzy:                                      {roc_auc_score(sem['ALVO_2025'], sem['prioridade']):.3f}")

Municípios sem facção em 2024: 549  |  passaram a ter em 2025: 129 (23.5%)

Entrada em 2025 por prioridade fuzzy (calculada só com dados até 2023):
               municipios  entraram
decisao_fuzzy                      
baixa                 352     0.131
média                 103     0.408
alta                   94     0.436

Entrada em 2025 pela decisão rígida:
                municipios  entraram
decisao_rigida                      
alta                    84     0.536
baixa                  465     0.181

AUC da probabilidade bayesiana para prever a entrada em 2025: 0.764
AUC da prioridade fuzzy:                                      0.751


## 3. Com os dados de 2024: a violência do ano anterior antecipa a chegada?

A pesquisa da 4ª edição foi feita entre nov/2024 e set/2025. Com o VDE de 2024, testamos duas hipóteses:

1. **Janela mais recente:** atributos de 2022–2024, em vez de 2021–2023, preveem melhor a entrada em 2025?
2. **Salto de violência:** um município cujos homicídios **subiram em 2024** (acima da média de 2022–23) tem mais chance de ganhar facção, como no caso de Iranduba (Sales et al., 2025)?

Modelo: regressão logística com log(população), taxas e presença de facção no estado; validação cruzada 5-fold só entre os municípios sem facção em 2024.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
sem_d = d[d["ALVO"] == 0].copy()
y_e = sem_d["ALVO_2025"].values
share_uf = sem_d["UF"].map(d.groupby("UF")["ALVO"].mean()).rename("share_uf")
eventos = ["homicidio", "tent_homicidio", "latrocinio", "mortes_esclarecer", "mandados"]
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)

def avaliar_janela(cols_extra=None, janela="2021_23"):
    X = pd.concat([np.log(sem_d[["pop_2022"]]), np.log1p(sem_d[[f"taxa_{e}_{janela}" for e in eventos]]), share_uf]
                  + ([sem_d[cols_extra]] if cols_extra else []), axis=1)
    r = cross_validate(make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
                       X, y_e, cv=cv5, scoring=["roc_auc", "balanced_accuracy"])
    return r["test_roc_auc"].mean(), r["test_balanced_accuracy"].mean()

sem_d["SALTO_2023"] = (sem_d["homicidio_2023"] > (sem_d["homicidio_2021"] + sem_d["homicidio_2022"]) / 2 + 1).astype(int)
sem_d["SALTO_2024"] = (sem_d["homicidio_2024"] > (sem_d["homicidio_2022"] + sem_d["homicidio_2023"]) / 2 + 1).astype(int)
linhas = [["atributos 2021–23", *avaliar_janela(janela="2021_23")],
          ["atributos 2022–24", *avaliar_janela(janela="2022_24")],
          ["atributos 2022–24 + salto em 2024", *avaliar_janela(["SALTO_2024"], janela="2022_24")]]
print(pd.DataFrame(linhas, columns=["modelo", "AUC", "acur. balanceada"]).round(3).to_string(index=False))
print()
for c in ["SALTO_2023", "SALTO_2024"]:
    t = sem_d.groupby(c)["ALVO_2025"].agg(municipios="size", entraram_2025="mean").round(3)
    print(f"{c} (homicídios acima da média dos 2 anos anteriores):"); print(t); print()

                           modelo   AUC  acur. balanceada
                atributos 2021–23 0.754             0.719
                atributos 2022–24 0.754             0.706
atributos 2022–24 + salto em 2024 0.755             0.702

SALTO_2023 (homicídios acima da média dos 2 anos anteriores):
            municipios  entraram_2025
SALTO_2023                           
0                  428          0.210
1                  121          0.322

SALTO_2024 (homicídios acima da média dos 2 anos anteriores):
            municipios  entraram_2025
SALTO_2024                           
0                  420          0.212
1                  129          0.310



**Leitura.** Usar a janela mais recente (2022–2024) **não melhora** a previsão da entrada de facção em 2025, e o salto de homicídios em 2024 também não acrescenta informação ao modelo. Municípios com salto em 2024 entraram mais (cerca de 31% contra 21%), mas o salto de **2023** tem praticamente o mesmo efeito. Ou seja, o salto não é um sinal específico da **chegada** da facção: é um sinal genérico de municípios violentos, que o modelo já captava pelas taxas.

O caso de Iranduba (homicídios saltando após a criação de uma facção) não aparece como padrão na escala dos 808 municípios. Isso é um resultado legítimo e vale ser relatado: o volume e o perfil da violência indicam **onde** a facção tende a estar, mas não **quando** ela chega. Uma hipótese é que a facção chegue antes de a violência subir, ou que o ano da pesquisa do Cartografias não coincida com o ano da chegada.

In [7]:
pp = np.linspace(0, 1, 101)
plt.figure(figsize=(6.5, 3.8))
for t, nome in [(-0.5, "homicídios caindo"), (0, "estável"), (1, "homicídios dobrando")]:
    plt.plot(pp, [avaliar(sistema, p, t) for p in pp], label=f"tendência: {nome}")
plt.plot(pp, np.where(pp >= 0.5, 100, 0), "k--", lw=1, label="limiar rígido 0,5")
plt.xlabel("P(facção) da rede bayesiana"); plt.ylabel("prioridade (0–100)")
plt.title("Saída fuzzy suave × limiar rígido"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figuras/etapa6_fuzzy_vs_rigido.png", dpi=150)

In [8]:
# Municípios sem facção em 2024 com prioridade fuzzy alta: candidatos à expansão
cand = sem.sort_values("prioridade", ascending=False).head(15)
cand[["uf", "municipio", "pop_2022", "p_faccao", "variacao_hom", "prioridade", "ALVO_2025"]].round(3)

,uf,municipio,pop_2022,p_faccao,variacao_hom,prioridade,ALVO_2025
734,TO,LAGOA DA CONFUSÃO,15288.0,0.394,1.857,83.254,0
539,PA,ORIXIMINÁ,68294.0,0.922,0.040,83.040,0
624,RO,JARU,50591.0,0.857,0.167,81.944,1
588,PA,TERRA SANTA,18782.0,0.356,1.250,81.629,1
262,MA,SANTA LUZIA,57635.0,0.784,0.000,81.482,0
149,MA,CAXIAS,156973.0,0.783,0.008,81.467,0
38,AM,CARAUARI,28742.0,0.380,1.167,81.291,1
597,PA,VISEU,58692.0,0.854,1.118,81.082,0
750,TO,NOVA OLINDA,10367.0,0.429,1.000,80.556,0
721,TO,FORMOSO DO ARAGUAIA,18881.0,0.429,1.000,80.556,1


## Discussão

- **A ordem das prioridades confere com o Cartografias 2024**: a proporção de municípios com facção, e em disputa, cresce de baixa para média e para alta.
- **O alerta antecipa parte da expansão de 2025.** Entre municípios que **não** tinham facção em 2024, a taxa de entrada em 2025 cresce com a prioridade calculada só com dados até 2023 (de cerca de 13% na baixa para cerca de 44% na alta). É uma validação fora do tempo, sem nenhum dado de 2025 no treino. O limiar rígido também separa bem, mas joga tudo em dois grupos; o fuzzy cria uma faixa intermediária que já concentra mais de 40% de entradas.
- **Cuidado ao interpretar as entradas.** O próprio Cartografias avisa que parte do aumento entre edições pode refletir a melhoria do método e mais operações policiais, e não só expansão real. 45 municípios "saíram" da lista entre 2024 e 2025, o que reforça esse alerta.
- **Interpretabilidade.** Cada regra é uma frase que um analista de inteligência entende e pode contestar.
- **Ética.** A saída é uma prioridade estatística para monitoramento, não uma acusação. O relatório não deve apresentar a lista de candidatos como municípios "dominados".

## Integração final do pipeline

VDE 2020–2024 + IBGE + Cartografias 2024 → **E1** árvore → **E2** regressão e logística → **E3** Bayes diagnóstico e transições → **E4** Naive Bayes + regimes ocultos (EM) → **E5** rede bayesiana: P(facção | porte, fronteira, sintomas) → **E6** fuzzy: prioridade de monitoramento → **validação com o Cartografias 2025**.